# Phase 2.6: Relational Integrity & Safe Join Validation
## Open University Learning Analytics Dataset (OULAD)

**Objective**: Empirically validate relationships between all seven OULAD tables, verify candidate primary and foreign keys, audit referential integrity, and evaluate safe join strategies to prevent data leakage and Cartesian explosions in ML modeling.

In [ ]:
import sys
sys.path.append('..')
import os
import json
import pandas as pd
import numpy as np

from src.relationship_validator import OULADRelationshipValidator

RAW_DIR = '../data/raw'
print('Libraries and modules imported successfully.')

### 1. Run Complete Validation Suite via `OULADRelationshipValidator`

In [ ]:
validator = OULADRelationshipValidator(raw_data_dir=RAW_DIR)
results = validator.run_all()
print('All relational integrity validations and join simulations executed.')

### 2. Enrollment Key Uniqueness & 1-to-1 Symmetry Audit

Evaluating the composite enrollment key `(code_module, code_presentation, id_student)` between `studentInfo.csv` and `studentRegistration.csv`.

In [ ]:
df_enrol = pd.DataFrame([results['enrollment_keys']])
display(df_enrol.T)

### 3. Assessment & Course Referential Integrity

- `assessments.csv` has 206 unique items across 22 presentations.
- 18 assessments (all final exams) have zero recorded submissions.
- 6,750 students (20.71%) have zero submissions in `studentAssessment.csv`.

In [ ]:
print('Assessment Validation Summary:')
for k, v in results['assessment_relationships'].items():
    print(f'  {k}: {v}')

print('\nCourse Presentation Validation Summary:')
for k, v in results['course_relationships'].items():
    print(f'  {k}: {v}')

### 4. VLE Referential Integrity & Inactivity Patterns

- 100% referential integrity between `studentVle.csv` and `vle.csv` (0 unmatched sites).
- 3,365 students (10.32%) have zero VLE clickstream interactions.

In [ ]:
for k, v in results['vle_relationships'].items():
    print(f'  {k}: {v}')

### 5. Join Simulations & Cardinality Analysis Table

In [ ]:
df_sim = pd.DataFrame(results['join_simulations'])
cols = ['join_name', 'expected_cardinality', 'left_rows', 'right_rows', 'result_rows', 'expansion_ratio', 'is_safe', 'notes']
display(df_sim[cols])

### 6. Relational Architecture Rules for Feature Engineering (Phase 5)

1. **Never perform unaggregated row-level joins** from student entities into `studentAssessment` or `studentVle`.
2. **Always aggregate transaction tables** to `(code_module, code_presentation, id_student)` before joining.
3. **Always use LEFT JOINs** starting from `studentInfo` to prevent discarding the 6,750 students with zero submissions and 3,365 students with zero VLE clicks.
4. **Never join `studentAssessment` directly to `studentVle`** across different grains (Cartesian explosion).